# 00 — 三代预训练数据清洗方法论概述

## 第一节：三代方法论演进

> **指标口径说明**（本 Notebook 中首次出现，后续引用同一口径）：
> - **数据保留率**：分子=Pipeline 最终输出的文档数，分母=Pipeline 接收的原始输入文档数。例如"30-40%"表示每 100 条原始输入，经 Pipeline 后保留 30-40 条。各论文的原始输入数据集不同（C4 输入约 6TB 原始文本, FineWeb 输入 96 个 CC dumps 约 100TB, DCLM 输入约 240B tokens），此处为各论文报告的典型保留率。
> - **7B MMLU**：在 7B 参数量的语言模型上，使用对应方法清洗后的数据训练，在 MMLU（Massive Multitask Language Understanding, 57 个学科的多选题）benchmark 上的准确率。分子=模型答对的题目数，分母=MMLU 总题目数（约 14,042 题）。

| | 第一代（Heuristic） | 第二代（Model-based） | 第三代（Hybrid） |
|---|---|---|---|
| 代表论文 | FineWeb, C4, Gopher (2020-2022) | DCLM (NeurIPS 2024) | Nemotron-CC (NVIDIA 2024) |
| 核心方法 | 人工设计规则 | fastText 分类器 + top-10% | 分类器集成 + Bypass + 改写 |
| 数据保留率 | 30-40%（口径见上） | **~10%** | **~38%** |
| 7B MMLU | ~55%（口径见上） | **~64%** | **~69%** |
| 核心优点 | 可解释、极快 | 能评估语义质量 | 高质量+高数量 |
| 核心缺陷 | 无法评估语义质量 | 90% 数据丢弃（=1 - 保留率 10%） | API 成本较高 |

方法论演进的核心驱动力：
- 第一代：规则 → 第二代：让模型学会"什么是高质量"
- 第二代：高质量但量少 → 第三代：打破 quality-quantity 的 false trade-off

### 1.1 第一代：Heuristic Filtering（规则过滤）

**代表论文**：FineWeb (HuggingFace 2024), C4 (Google 2019), Gopher (DeepMind 2021)

**核心思想**：用人工设计的统计规则逐步过滤低质量内容。每条规则针对一类典型噪声。

**Pipeline 执行顺序**（按 FineWeb 实际顺序，`src/gen1/pipeline.py`）：

> **口径约定**：下表"论文参考值"列的默认口径为**条件过滤率**（分子=该步丢弃的文档数，分母=该步输入文档数）。如某行口径不同，在"口径说明"列标注。

| 步骤 | 过滤器 | 过滤目标 | 关键参数 | 论文来源 | 论文参考值 | 口径说明 |
|------|--------|---------|---------|---------|-----------|---------|
| Step 0 | URL 去重 | 完全相同 URL 的重复文档 | 精确 URL 匹配 | RefinedWeb | ~10-14% | 条件过滤率；RefinedWeb 多 crawl 聚合场景，单 crawl 更低 |
| Step 1 | URL Filter | 成人/垃圾/恶意网站 | 黑名单域名 + 关键词 | FineWeb | ~2.1% | 条件过滤率；FineWeb 使用 UT1 黑名单，分母=URL 去重后文档数 |
| Step 2 | Language Filter | 非目标语言文档 | fastText-langid, 置信度 ≥ 0.65 | C4 / RefinedWeb | C4: 存活率 ~38%；RefinedWeb: 非英文 ~60% | **注意口径不同**：C4 为整体存活率（分子=英文文档, 分母=C4 原始输入总量，含去重）；RefinedWeb 为条件过滤率 |
| Step 3 | Quality Filter | 不像正常文章的文档 | Gopher + C4 + FineWeb 规则组合 | Gopher / C4 / FineWeb | ~20-30%（估算；子规则详见下表） | 估算值：各子规则串行执行且有重叠，合计低于各项之和。Gopher Table A1 未报告质量过滤合计值 |
| Step 4 | Repetition Filter | 内容高度重复的文档 | 重复行/段落占比 + N-gram 重复比例（多粒度阈值） | Gopher | ~10-15% | 条件过滤率；分母=通过质量过滤后的文档数；Gopher Table A1 |
| Step 5 | PII Filter | 包含隐私信息的文档 | Email/Phone/IP/SSN 正则 → 脱敏 | RefinedWeb | <1%（≈ 0%） | PII 做替换脱敏，不删除文档，条件过滤率 ≈ 0% |

---

**Step 3 Quality Filter 子规则详解**（最复杂的一步）：

> **口径约定**：下表为各论文报告的**单条规则独立过滤率**（分子=仅该规则命中的文档数, 分母=进入质量过滤步骤的总文档数）。由于规则串行执行且存在重叠（同一文档可被多条规则同时命中），各规则独立过滤率之和 ≠ 整体过滤率。此外，Gopher Table A1 定义了 8+ 条质量规则，下表仅列出影响最大的子规则。

| 规则类别 | 具体规则 | 阈值 | 过滤目标 | 论文参考值 | 口径说明 |
|----------|---------|------|---------|-----------|---------|
| **Gopher** | 词数范围 | 50 ≤ words ≤ 100K | 过短/过长 | ~6.8% | Gopher Table A1, MassiveWeb 数据集 |
| **Gopher** | 平均词长 | 3-10 字符 | 编码垃圾/非自然文本 | ~3.2% | 同上 |
| **Gopher** | 字母字符占比 | ≥ 80% | 纯数字/符号/编码垃圾 | ~2.5% | 同上 |
| **Gopher** | 符号-词比例 | ≤ 0.1 | 符号密集文档（#、…等） | ~1.5% | 同上 |
| **Gopher** | 停用词数量 | ≥ 2 | 非自然语言（代码、表格） | ~5% | 同上 |
| **Gopher** | 省略号行占比 | ≤ 30% | 截断内容、SEO 摘要 | ~0.5% | 同上 |
| **C4** | 句末标点行比例 | ≥ 30% | 非完整句子（列表、菜单） | 论文未单独报告 | C4 整体 pipeline 保留率 ~12.5%（分子=C4 输出 750GB, 分母=CC 原始约 6TB, 含全部步骤；Raffel et al. 2019 §2.2），此规则为 C4 过滤力度最大的单条 |
| **C4** | JavaScript 检测 | 包含 "javascript" → 过滤 | 未清除的代码片段 | ~1% | Raffel et al. 2019 §2.2 |
| **C4** | 最少行数 | ≥ 3 行 | 超短片段 | ~5% | Raffel et al. 2019 §2.2 |
| **FineWeb** | 字母词占比 | ≥ 60% | 乱码、编码错误 | ~2% | FineWeb 消融实验 §3.2 |

> **备注——"字母字符占比"（alpha ratio）是什么？**
>
> 口径：分子 = 文档中字母字符数（`a-z`, `A-Z`），分母 = 文档总字符数（含空格、数字、标点等）。正常英文文章约 80-85%（空格和标点占 ~15-20%），Gopher 要求 ≥ 80%。以下三类文档会显著低于此阈值：
> - **数字密集页面**：如纯数据表格 `"2024-03-01 | 09:30 | 182.63 | +1.2% | 2,340,000"` → 几乎全是数字、日期和符号，字母占比极低
> - **编码乱码**：网页字节被错误解码产生的无意义字符。如 URL 编码残留 `"%E4%BD%A0"` 中 `%` 和十六进制数字占绝大多数，字母极少；又如中文 UTF-8 被误读为 Latin-1 产生 `"Ã¤Â½Â "`，其中 `½`（分数符号）、`¤`（货币符号）等都不是字母
> - **代码/样式残留**：如 WET 提取未完全清除的 CSS `"{margin:0;padding:0;font-size:12px}"` → 大括号、冒号、分号、数字大量堆积，字母被稀释

---

**Step 4 Repetition Filter 子规则详解**：

> **口径约定**：下表"论文参考值"为**条件过滤率**（口径同 Step 3）。数据来源：Gopher Table A1（MassiveWeb 数据集）。
>
> **阈值的口径说明**：各阈值中的"占比"均为字符级别——分子=符合重复条件的字符总数，分母=文档总字符数。超过阈值的文档被整篇过滤。

| 规则 | 阈值 | 过滤目标 | 论文参考值 | 口径说明 |
|------|------|---------|-----------|---------|
| 重复行占比 | ≤ 30% | 模板化页脚/侧栏 | ~2-3% | 分子=重复出现的行的字符总数，分母=文档总字符数 |
| 重复段落占比 | ≤ 30% | 复制粘贴内容 | ~1-2% | 分子=重复出现的段落的字符总数，分母=文档总字符数 |
| Top 2-gram 占比 | ≤ 20% | 机械重复短语 | ~3-5% | 分子=出现频率最高的 2-gram 覆盖的字符数，分母=文档总字符数 |
| 重复 5-10gram 占比 | ≤ 10-15% | 长段落复制 | ~5-8% | 分子=出现 ≥2 次的 n-gram 覆盖的字符数，分母=文档总字符数 |

> **Gopher 重复过滤的执行逻辑——多规则并行检测，任一命中即过滤**：
>
> 对每篇文档，上述四条规则**各自独立计算**（不是串行地逐条淘汰），然后取并集：只要任意一条规则超过阈值，整篇文档就被过滤。
>
> 之所以需要多条规则，是因为不同粒度各有盲区：
> - **行级 / 段落级**：能抓整块复制（如页脚版权声明、侧栏导航），但抓不到跨行的重复短语
> - **N-gram 级**：能抓分散的重复短语（如关键词堆砌 "buy cheap shoes buy cheap shoes..."），但抓不到整段复制
>
> 多粒度组合才能覆盖不同类型的重复模式。

---

**第一代的核心局限**：
- 规则只能过滤"明显的垃圾"，无法区分"平庸内容"和"高质量内容"
- 一篇语法正确但毫无信息量的营销文案，可以通过所有规则
- 规则之间存在不可组合性（详见术语表第 3 条）

### 1.2 第二代：Model-based Filtering（分类器过滤）

**代表论文**：DCLM (NeurIPS 2024), FineWeb-Edu (HuggingFace 2024)

**核心思想**：训练一个 fastText 二分类器（工作原理详见 NB03 Cell Group A），让它学会区分"高质量文本"和"低质量文本"，然后只保留分类器认为最高质量的 top-10% 文档。

**Pipeline 流程**（`src/gen2/pipeline.py`）：

```
输入文档 → fastText 分类器打分 → 按分数排序 → 保留 top-10% → 输出
```

> **为什么只用一个分类器就够？** Gen2 的核心操作是"打分 → 排序 → 取 top-10%"，只需要一个分类器给出一致的排序即可。单个 fastText 处理 TB 级数据只需几分钟，7B MMLU 已达 64%（DCLM Table 3 论文值，详见下方对比表）。Gen2 的瓶颈不在分类器数量，而在于 top-10% 策略本身——90% 数据被丢弃且分布偏移（详见本节末尾"核心局限"）。多分类器集成是 Gen3 的创新，目的不是提高打分精度，而是扩大高质量数据的覆盖面（§1.3 创新 1）。

**分类器训练配置**：

| 项目 | 本项目配置 | DCLM 论文配置 |
|------|-----------|--------------|
| 正样本 | Wikipedia 摘要（详见 §2.2） | Wikipedia + OpenHermes（论文同时测试两种） |
| 负样本 | 原始 CC WET 随机样本 | DCLM-Pool 随机样本（~240B tokens） |
| 模型 | fastText | fastText（论文最优方案） |
| 超参 | dim=64, wordNgrams=2 | dim=64, wordNgrams=2（DCLM Table 2 最优） |

> **为什么正样本只用 Wikipedia？** DCLM 论文测试了三种正样本方案（详见 §2.2），结果 Wikipedia 单独使用时 7B MMLU 最高（64.0%），优于 OpenHermes（63.2%）和两者混合（63.8%）。此外 Wikipedia 公开可获取、无版权争议、可复现性强，是学术界的标准选择。

**为什么选 fastText 而非 BERT？**

> **口径说明**：AUC = ROC 曲线下面积，取值 0~1，1 为完美分类（分子=TPR, 分母=FPR 的积分面积）。MMLU 口径同 §1 首表。速度为处理 50K 文档的端到端时间，来源 DCLM 论文 §3.1。

| 模型 | 速度（50K 文档） | AUC | 7B MMLU | 适合场景 |
|------|-----------------|-----|---------|---------|
| **fastText** | < 1 分钟 | ~0.85 | ~64% | 大规模清洗（TB 级） |
| BERT | 数小时（慢 ~100x） | ~0.92 | ~66% | 小量精标数据 |

> 来源：DCLM §3.1（速度对比）、FineWeb-Edu 论文（BERT AUC）、DCLM Table 3（MMLU；绝对差 2 个百分点）。DCLM 结论：速度-效果权衡下 fastText 最优。

**阈值选择实验**（DCLM 论文 Table 3 结论）：

> **口径说明**：保留比例 = 分子: 保留的文档数 / 分母: 打分后的总文档数。MMLU 口径同 §1 首表。数据量为 DCLM-Pool 中按保留比例截取后的估算 token 数。

| 保留比例 | DCLM 论文 7B MMLU | DCLM 论文数据量 | 结论 |
|---|---|---|---|
| top 5% | 62.1% | ~12B tokens | 数据不足导致模型欠拟合 |
| **top 10%** | **64.0%** | **~24B tokens** | **DCLM 最优点：MMLU 最高** |
| top 20% | 61.5% | ~48B tokens | 低质量数据稀释了训练信号 |
| top 50% | 58.3% | ~120B tokens | 接近未做质量过滤的效果 |

**循环偏差防范**（本项目的关键设计）：

Pipeline 分类器负责过滤数据，但我们还需要一个方法来**评估过滤后的数据质量到底好不好**。如果直接用 Pipeline 分类器自己来打分评估，就好比"自己出题自己考试"——分数必然很高，但不说明数据真的好。

因此本项目额外训练了一个**评估分类器**（Evaluation Classifier）：它和 Pipeline 分类器做同样的事情（给文档打质量分），但**故意使用不同的超参数**，确保两者是独立的"判卷人"。评估分类器**只用于打分**（在 Notebook 中衡量各代数据质量），不参与 Pipeline 的过滤决策。

> **评估分类器需要自行训练**，不是可下载的现成模型。本项目中由 `scripts/run_gen2.py` 执行时一并完成——使用相同的训练数据来源（Wikipedia 正样本 + CC WET 负样本），但故意设置不同的超参数。训练只需几秒钟（fastText 训练极快），训练完成后模型保存在 `results/quality_scores/eval_classifier.bin`。

| | Pipeline 分类器 (Gen2) | 评估分类器 |
|---|---|---|
| 正样本 | Wikipedia | Wikipedia（独立训练集） |
| dim | 64 | 32 |
| wordNgrams | 2 | 1 |
| 用途 | 过滤数据 | 只打分，不过滤 |

> **dim 和 wordNgrams 是什么？** 这两个是 fastText 的核心超参数（原理详见 NB03）：
> - **dim**（向量维度）：fastText 把每个词/n-gram 映射为一个数字向量，dim 决定向量的长度。dim=64 表示 64 个数字，dim=32 表示 32 个数字。维度越高，模型表达能力越强，但也越容易和另一个模型"学到相同的模式"
> - **wordNgrams**（词组长度）：fastText 除了看单个词，还可以看相邻词的组合。wordNgrams=2 表示同时考虑两个词的组合（如 `"the cat"`），wordNgrams=1 表示只看单个词
>
> **为什么要故意设不同？** dim=64 的模型能捕捉更精细的语义模式，dim=32 的模型相对粗糙；wordNgrams=2 能看到短语，wordNgrams=1 只看单词。两者学到的"高质量"判断标准会有差异，这正是我们想要的——评估结果不会因为模型相同而产生虚假的高分。

> 循环偏差防范设计参考 DCLM 论文 §4.2 的讨论：评估必须独立于 pipeline，否则会产生"自证"的假高分。详见术语表第 6 条。

**第二代的核心局限**：
- **90% 数据被丢弃**：top-10% 保留率意味着丢掉 90% 的数据（分子=丢弃文档数, 分母=总文档数）
- **分布偏移**：分类器偏好"百科风格"，导致保留的数据缺乏多样性（对话/新闻/创作类内容被低估）
- **不适合长 Token Horizon 训练**：Token Horizon = 预训练消耗的总 token 数（详见术语表第 5 条）。Llama 3 的 Token Horizon 为 15T tokens，top-10% 过滤后的可用数据远远不够

### 1.3 第三代：Hybrid Pipeline + Data Recovery

**代表论文**：Nemotron-CC (NVIDIA 2024), GneissWeb (IBM 2024)

**为什么直到 2024 年才出现？** 分类器集成、条件路由、LLM 改写——单个技术都不新鲜，但三个前提条件在 2024 年才同时成熟：

1. **数据量瓶颈 2024 年才真正出现**：GPT-3（2020）只需 300B tokens，Gen1 清洗完全够用；Llama 3（2024）需要 **15T tokens**，Gen2 的 top-10% 策略丢掉 90% 数据，根本不够。在数据不是瓶颈的时代，没人有动力改进 Gen2。
2. **Gen2 自身 2024 年才被系统化**：DCLM 是 NeurIPS 2024 论文。在此之前，学术界对分类器质量过滤没有系统性的比较和结论。Gen3 必须先清楚 Gen2 的瓶颈在哪（90% 丢弃 + 分布偏移），才能针对性地解决。
3. **LLM 改写成本 2024 年才可行**：合成改写需要大规模调用 LLM。2022 年 GPT-3 API 价格极高，TB 级改写不现实。2024 年 Mistral 7B 等开源模型出现后，改写成本下降了约 100 倍。

**核心思想**：打破"质量高 vs 数量多"的 false trade-off。通过三项创新，在保持质量的前提下将数据保留率从 ~10% 提升至 ~38%（口径同 §1 首表）。

**Pipeline 流程**（`src/gen3/pipeline.py`）：

```
输入文档
  │
  ├─ 分类器集成打分（创新 1）
  │
  ├─ 高质量 (score ≥ 0.7) ────────→ 直接保留（跳过规则过滤）── 创新 2
  │
  ├─ 中等质量 (0.3 ≤ score < 0.7) → 应用 Heuristic 过滤 → 通过则保留，未通过则丢弃
  │
  ├─ 低质量 (0.1 ≤ score < 0.3) ──→ LLM 改写 ── 创新 3 → 质量验证 → 保留/丢弃
  │
  └─ 极低质量 (score < 0.1) ──────→ 直接丢弃（无改写价值）
```

---

**创新 1：分类器集成（Classifier Ensembling）**

> **口径说明**：权重为加权平均时各分类器的贡献比例，三者之和 = 1.0。本项目参考 Nemotron-CC 的多分类器集成思路，具体权重为本项目设计（Nemotron-CC 论文未公开具体权重值）。

| 分类器 | 类型 | 正样本来源 | 正样本偏好 | 权重 | 训练方式 |
|--------|------|-----------|-----------|------|---------|
| fasttext_dclm | fastText (dim=64) | Wikipedia 摘要 | 百科/学术风格 | 0.4 | 自行训练（由 `scripts/run_gen2.py` 训练，Gen3 复用） |
| fasttext_edu | fastText (dim=64) | Cosmopedia 合成教科书 | 教育/教科书风格 | 0.4 | 自行训练（由 `scripts/run_gen3.py` 训练） |
| tfidf_lr | TF-IDF + LogisticRegression | Wikipedia 摘要 | 综合质量 | 0.2 | 自行训练（由 `scripts/run_gen3.py` 训练） |

> **三个分类器都需要自行训练**，不是可下载的现成模型。训练过程均为几秒到几十秒（fastText 和 TF-IDF+LR 都是轻量级模型）。正样本分别来自 Wikipedia（§2.2）和 Cosmopedia（§2.3），负样本从 CC WET 原始数据中随机抽取。

> **Nemotron-CC 论文的集成成员**：论文使用 3 个分类器——(1) DCLM fastText，(2) FineWeb Mixtral Edu Classifier，(3) FineWeb Nemotron-4 Edu Classifier。后两者是在 467K FineWeb 文本上用 LLM（Mixtral 和 Nemotron-4-340B）标注教育价值分后训练的。本项目用 Cosmopedia 合成教科书替代 FineWeb LLM 标注数据，避免引入 FineWeb（已清洗数据）的循环偏差——详见 §2.3。

> **为什么额外加入 TF-IDF + LR？** Nemotron-CC 的集成成员全部是 fastText——架构相同，只有训练数据不同。这意味着它们只在"什么算高质量"上有分歧，但**提取文本特征的方式完全一样**（都是词/子词的稠密向量求平均）。如果 fastText 的特征提取方式对某类文档存在系统性盲区（例如关键词分布异常但语义向量正常的文档），所有 fastText 成员都会同时漏掉。
>
> TF-IDF + LR 用完全不同的方式理解文本：它不学习向量，而是直接统计每个词出现的频率和稀有程度。这给集成带来了**架构多样性**——不仅"质量标准"不同，连"看文本的方式"也不同。集成学习的核心原则就是：成员之间差异越大，集成效果越好。
>
> 权重设为 0.2（而非 1/3）是因为 TF-IDF 在本项目中是实验性补充，主体仍以 Nemotron-CC 的 fastText 方案为主。实际效果对比见 NB04。

> **三种分类器的设计差异**：
>
> 三者均为"文本表征 → 二分类器"架构，差异体现在两个维度：
> 1. **训练样本不同**（决定"高质量"的定义边界）：
>    - DCLM：正样本为 Wikipedia 百科（通用性强），偏好事实性、规范性文本
>    - EDU：正样本为 Cosmopedia 合成教科书（OpenStax 教材风格），偏好有教学价值、结构清晰的文本
> 2. **文本表征方法不同**（决定特征提取的粒度）：
>    - fastText：学习词和字符 n-gram 的稠密向量，粒度更细，能处理未见词（通过子词组合）
>    - TF-IDF：直接计算词频-逆文档频率的稀疏统计，粒度为词级别，对词汇覆盖率和关键词分布敏感

**集成策略**：Union（任一分类器认为高质量 → 判为高质量）

**为什么需要集成？** 单一分类器都有覆盖盲区。例如技术博客可能被"百科风格"分类器低估，却被"教育类"分类器高估。集成扩大了高质量数据的覆盖面。

Nemotron-CC 论文数据：集成后 unique token 覆盖量比单一分类器多 **+28%**（分子=集成后 unique token 数 - 单一分类器 unique token 数, 分母=单一分类器 unique token 数；即相对增长率），同时 7B MMLU 不降（口径同 §1 首表）。

---

**创新 2：条件性 Heuristic Bypass**

> **口径说明**：路由条件中的"分数"为分类器集成的加权平均输出，取值 0~1。阈值 0.7/0.3/0.1 参考 Nemotron-CC 论文的分桶策略（论文使用 HQ/MQ/LQ 三桶，具体阈值未公开，本项目 0.7/0.3/0.1 为经验设定）。

| 路由条件 | 处理方式 | 原理 | Nemotron-CC 论文参考 |
|----------|---------|------|---------------------|
| 集成分数 ≥ 0.7 | **跳过** heuristic，直接保留 | 高分文档已被分类器确认高质量，规则过滤是多余的，还可能误杀 | Nemotron-CC HQ 桶：直接保留 |
| 0.3 ≤ 分数 < 0.7 | 应用 heuristic 过滤，**未通过则丢弃** | 中等分数需要规则做二次验证；通过 → 保留，未通过 → 丢弃 | Nemotron-CC MQ 桶：应用 heuristic |
| 0.1 ≤ 分数 < 0.3 | 送去 LLM 改写 | 内容有回收价值但表达质量差，LLM 能改善表达 | Nemotron-CC LQ 桶：synthetic rephrasing |
| 分数 < 0.1 | 直接丢弃 | 质量极低，连改写也无法救回 | — |

> **为什么 MQ 未通过 heuristic 的文档是丢弃而非改写？**
>
> Heuristic 规则检测的是**结构性问题**（内容重复、编码乱码、格式异常），而 LLM 改写擅长改善的是**表达质量**（把写得差的内容用更好的方式重新表述）。一篇文档如果因为大量重复内容、编码垃圾或格式混乱被 heuristic 拒绝，LLM 改写后这些结构性问题仍然存在——重复的信息改写后还是重复的。
>
> 相比之下，LQ 文档（0.1-0.3）是因为**内容/表达差**被分类器判为低质量，但其中可能包含有价值的信息。LLM 可以提取核心信息并用更好的方式重新表述，这正是 LLM 的强项。
>
> Nemotron-CC 论文也采用相同的设计：MQ fail → discard，不送去改写。论文未对此做消融实验，但从成本效益角度看：改写有 API 成本，优先改写"信息有价值但表达差"的 LQ 文档比改写"有结构性问题"的 MQ fail 文档收益更高。

**Bypass 误杀率**（Nemotron-CC 的关键发现）：被 heuristic 误杀的文档中，18.1% 实际上是高质量内容。口径：分子=被 heuristic 拒绝但被分类器判为高质量（score ≥ HQ 阈值）的文档数，分母=被 heuristic 拒绝的总文档数（Nemotron-CC 论文 §4.3）。

---

**创新 3：合成数据改写（Synthetic Rephrasing）**

> **口径说明**：改写成功率 = 分子: 改写后通过质量验证的文档数 / 分母: 送去改写的文档总数。
>
> **质量验证使用的分类器**：改写后的文本用**同一套分类器集成**（即创新 1 中的 3 个分类器组合）重新打分。这保证了打分标准一致——路由阶段和验证阶段用同一把尺子衡量质量。
>
> **为什么质量门禁是 0.4 而非 0.7？** 两个阈值的用途不同：
> - **0.7（HQ bypass 阈值）**= "文档质量已经很高，可以跳过 heuristic" — 这是对**自然文档**的最高标准
> - **0.4（改写后质量门禁）**= "改写后是否有足够提升，值得保留" — 这是对**合成数据**的最低标准
>
> 被送去改写的文档原始集成分数在 0.1-0.3。如果改写后分数达到 0.4+，意味着有 0.1-0.3 的绝对提升，代表 LLM 改写产生了实质性的质量改善。合成数据不需要达到自然高质量文档的水平（0.7+），它的价值在于增加训练数据的多样性和覆盖面——Nemotron-CC 通过这种方式将可用 unique token 数量增加了 4 倍。

| 步骤 | 说明 | 论文参考值 | 口径/来源 |
|------|------|-----------|----------|
| 1. 筛选 | 从低质量文档中选集成分数在 0.1-0.3 的（有改写价值） | 筛选 LQ 桶中 score 靠上的文档 | Nemotron-CC 论文 |
| 2. 改写 | 调用 LLM API 改写（保留核心信息，提升表达质量） | Mistral-7B-Instruct 改写 | Nemotron-CC 论文 |
| 3. 质量验证 | 改写后用**分类器集成**重新打分，集成分数 ≥ 0.4 才保留 | 成功率 ~70-80% | Nemotron-CC 论文 §5 |
| 4. 异常检测 | 检查 Perplexity 是否合理（防止 LLM 幻觉） | PPL 异常阈值过滤 | Nemotron-CC 论文 |

**本项目配置**：使用 Anthropic Claude Opus API（`configs/api_config.yaml`），与 Nemotron-CC（Mistral-7B-Instruct）不同——Opus 模型能力更强，改写质量更高。

**第三代的核心取舍**：
- **优点**：数据保留率从 ~10% 提升至 ~38%（口径同 §1 首表），打破 quality-quantity trade-off
- **代价**：API 成本（改写）、计算复杂度（集成 + 路由）、改写质量不确定性

### 1.4 三代方法对比：预期效果

基于各代表论文报告的数据，三代方法的预期效果对比如下。实际实验结果见 NB06（跨代对比分析）。

**预期数据保留率对比**：

> **口径说明**：保留率 = 分子: Pipeline 最终输出文档数 / 分母: Pipeline 输入文档数。各论文输入数据集不同（详见 §1 首表），此处为论文报告的典型值。

| 指标 | Gen1 Heuristic | Gen2 Model-based | Gen3 Hybrid |
|------|---------------|-----------------|-------------|
| 论文预期保留率 | 30-40% | ~10% | ~38% |
| 代表论文 | FineWeb / C4 / Gopher | DCLM | Nemotron-CC |
| 预期 7B MMLU | ~55% | ~64% | ~69% |

**Gen1 各步预期过滤率**（基于论文报告值）：

> **口径说明**：均为条件过滤率（分子=该步丢弃数, 分母=该步输入数），来源见 §1.1 各步详表。注意：论文报告值基于各自的输入数据集，与本项目使用的 CC WET 原始数据特征不同（如非英文比例、噪声水平），实际过滤率会有差异。

| 步骤 | 过滤器 | 论文预期条件过滤率 | 论文来源 |
|------|--------|-------------------|---------|
| Step 0 | URL 去重 | ~10-14% | RefinedWeb（多 crawl 聚合场景） |
| Step 1 | URL Filter | ~2.1% | FineWeb |
| Step 2 | Language Filter | ~60%（非英文） | RefinedWeb |
| Step 3 | Quality Filter | ~20-30%（估算，详见 §1.1） | Gopher / C4 / FineWeb |
| Step 4 | Repetition Filter | ~10-15% | Gopher Table A1 |
| Step 5 | PII Filter | ≈ 0%（脱敏不删除） | RefinedWeb |

**Gen3 预期路由分布与保留率**（基于 Nemotron-CC 论文）：

> **口径说明**：
> - 分配比例 = 分子: 落入该分数区间的文档数 / 分母: Gen3 Pipeline 总输入文档数
> - 桶内保留率 = 分子: 该桶最终保留的文档数 / 分母: 落入该桶的文档数
> - 对总保留率贡献 = 分配比例 × 桶内保留率
> - 阈值来源见 §1.3

| 分数区间 | 桶名称 | 预期分配比例 | 处理方式 | 桶内保留率 | 对总保留率贡献 |
|----------|--------|------------|---------|-----------|-------------|
| ≥ 0.7 | 高质量 (HQ) | ~15-20% | 直接保留，跳过 heuristic（创新 2） | ~100% | ~15-20% |
| 0.3 ~ 0.7 | 中等质量 (MQ) | ~25-35% | 应用 heuristic 过滤；通过→保留，未通过→丢弃 | ~50-70% | ~13-25% |
| 0.1 ~ 0.3 | 低质量 (LQ) | ~20-30% | 全部送去 LLM 改写，改写后集成分数 ≥ 0.4 → 保留 | ~70-80%（改写成功率） | ~14-24% |
| < 0.1 | 极低质量 | ~20-30% | 直接丢弃（无改写价值） | 0% | 0% |
| **合计** | | **100%** | | | **~42-69%** |

> **本项目配置**：所有 LQ 文档均全量送去改写（`rewrite_count: 99999`），不设改写预算上限。改写使用 Claude Opus API（详见 §2.5），具体改写量取决于运行后 LQ 桶的实际文档数。

> **各桶"丢弃"的具体来源**：
> - HQ 桶：几乎不丢弃（分类器已确认高质量）
> - MQ 桶丢弃 = heuristic 规则拒绝（结构性问题：内容重复/编码乱码/格式异常）
> - LQ 桶丢弃 = 改写后集成分数仍 < 0.4（改写未能产生实质性改善）+ API 调用失败
> - 极低质量桶：全部丢弃（连改写也无法救回）

**Gen3 关键预期指标**：

| 指标 | 论文预期值 | 口径说明 | 论文来源 |
|------|-----------|---------|---------|
| Bypass 误杀率 | ~18.1% | 分子=被 heuristic 拒绝但分类器判为高质量的文档数 / 分母=被 heuristic 拒绝的总文档数 | Nemotron-CC §4.3 |
| 改写成功率 | ~70-80% | 分子=改写后集成分数 ≥ 0.4 的文档数 / 分母=送去改写的文档总数 | Nemotron-CC §5 |
| 集成覆盖增量 | +28% unique tokens | 分子=集成后 unique tokens - 单分类器 unique tokens / 分母=单分类器 unique tokens（相对增长率） | Nemotron-CC |

**预期核心假设**（待 NB02-NB04 和 NB06 验证）：
1. **质量递增**：Gen2 quality_score > Gen1 > 原始数据（DCLM 核心结论）
2. **数量突破**：Gen3 保留率 ≈ Gen2 的 3-4 倍，quality_score 不降（Nemotron-CC 核心结论）
3. **Bypass 有效性**：Heuristic 对高分文档存在显著误杀，Bypass 机制能有效回收这部分数据

### 1.5 本项目实验设计

**研究问题**：在同一份原始数据上，三代预训练数据清洗方法（Heuristic → Model-based → Hybrid）的过滤效果差异有多大？质量提升能否量化？

**实验控制**（保证可比性）：

| 控制变量 | 说明 | 实现方式 |
|---------|------|---------|
| 输入数据 | Gen2/Gen3 在 Gen1 输出上运行，保证比较基准一致 | Gen1 输入 CC WET 原始数据；Gen2/Gen3 输入 Gen1 的输出（级联式架构，详见 §2.1） |
| 随机种子 | 所有随机操作（采样/shuffle）固定种子 | `configs/run_config.yaml` 中 `random_seed: 42` |
| 评估方法 | 同一套独立评估体系打分 | 评估分类器与 Pipeline 分类器超参不同（§1.2），避免循环偏差 |
| 数据规模 | 两档运行模式，规模由配置控制 | `smoke_test`（12K 条）/ `full_run`（100K 条），代码不硬编码 |

**自变量**（三代方法的核心差异）：

| | Gen1 | Gen2 | Gen3 |
|---|---|---|---|
| 过滤机制 | 规则（Heuristic） | 分类器 + top-10% | 分类器集成 + Bypass + 改写 |
| 质量判断依据 | 统计特征（词数/重复率等） | 学习到的"百科风格"分布 | 多源正样本集成覆盖 |
| 数据回收 | 无 | 无 | LLM 改写低质量文档 |

**因变量**（五维评估指标）：

| 维度 | 指标 | 工具 | 为什么需要这个维度 |
|------|------|------|------------------|
| 规模 | 文档数、词数、保留率 | 直接统计 | 衡量过滤强度 |
| 质量 | KenLM Perplexity、独立分类器打分 | KenLM Wikipedia 5-gram、评估分类器（dim=32） | 衡量文本规范性和语义质量 |
| 语言 | 英文占比、语言置信度 | fastText lid.176.bin | 确认语言过滤有效性 |
| 多样性 | N-gram unique ratio、域名 Shannon Entropy | 自定义统计 | 检测分布偏移（§术语表 7） |
| 毒性 | Detoxify 分数 | Detoxify 预训练模型 | 确认过滤不引入毒性偏差 |

**核心假设**（待 NB02-NB04 和 NB06 验证）：
1. **质量递增**：Gen2 quality_score > Gen1 > 原始数据
2. **数量突破**：Gen3 保留率 ≈ Gen2 的 3-4 倍，quality_score 不降
3. **Bypass 有效性**：Heuristic 对高分文档存在显著误杀

**已知局限**：
- 代理指标（质量分类器打分、PPL）只能间接反映数据对模型训练的效果，无法替代端到端 benchmark 验证
- 评估分类器虽与 Pipeline 分类器超参不同，但仍使用相同来源的正样本（Wikipedia），独立性有限
- smoke_test 模式（12K 条）经过多轮过滤后样本量较小，统计稳定性受限
- 可选的端到端验证（NB09：GPT-2 125M Proxy Model + benchmark）可部分弥补代理指标的局限，但 125M 模型规模远小于实际预训练场景

## 第二节：数据源与模型详解

本项目涉及的外部依赖分为两类：**数据源**（Pipeline 的输入和分类器训练数据）和**模型**（外部预训练模型、LLM API、项目自行训练的模型）。

### 总览

#### A. 数据源

| 名称 | 用途 | 使用阶段 | 规模 |
|------|------|---------|------|
| CC WET 原始数据 | Gen1 Pipeline 的输入（原始互联网文本） | Gen1 Pipeline | 12K~100K 条 |
| Gen1 输出 | Gen2 / Gen3 Pipeline 的输入（经 Gen1 过滤后的文本） | Gen2/Gen3 Pipeline | Gen1 保留的文档数 |
| Wikipedia 摘要 | fasttext_dclm / tfidf_lr / 评估分类器的正样本 | Gen2/Gen3 训练阶段 | 5,000 条 |
| Cosmopedia (OpenStax) | fasttext_edu 的正样本 | Gen3 训练阶段 | 5,000 条 |

> **数据流**：CC WET → Gen1 → gen1_output.jsonl → Gen2/Gen3 各自独立处理。Gen2 和 Gen3 不直接读取原始 CC WET，而是在 Gen1 规则过滤后的数据上进一步操作。这也是论文中的标准做法（如 DCLM 在 Dolma/FineWeb 基线上应用分类器）。

#### B. 外部预训练模型

| 名称 | 用途 | 使用阶段 | 来源 |
|------|------|---------|------|
| fastText lid.176.bin | 语言检测（176 种语言） | Gen1 Pipeline Step 2 | Facebook Research |
| KenLM Wikipedia EN | Perplexity 代理指标（**不参与过滤**） | 评估 Notebook（NB02/NB05） | HuggingFace `edugp/kenlm` |
| Claude Opus API | Gen3 低质量文档改写 | Gen3 Pipeline Step 3 | Anthropic |

#### C. 项目自行训练的模型

| 名称 | 训练脚本 | 正样本 | 超参数 | 使用阶段 |
|------|---------|--------|--------|---------|
| fasttext_dclm | `run_gen2.py` | Wikipedia | dim=64, wN=2 | Gen2 过滤 + Gen3 集成 |
| fasttext_edu | `run_gen3.py` | Cosmopedia | dim=64, wN=2 | Gen3 集成 |
| tfidf_lr | `run_gen3.py` | Wikipedia | 50K features | Gen3 集成 |
| eval_classifier | `run_gen2.py` | Wikipedia | dim=32, wN=1 | **仅评估**（NB01-NB07） |

#### D. 按 Pipeline 阶段的依赖清单

| 阶段 | 数据输入 | 外部模型 | 项目训练的模型 | 主要输出 |
|------|---------|---------|--------------|---------|
| **Gen1** | CC WET 原始数据 | fastText lid.176.bin（语言检测） | 无 | gen1_output.jsonl |
| **Gen2** | Gen1 输出 | 无 | fasttext_dclm（过滤用） | gen2_output.jsonl |
| **Gen3** | Gen1 输出 | Claude Opus API（改写用） | fasttext_dclm + fasttext_edu + tfidf_lr（集成打分） | gen3_output.jsonl |
| **评估** | 各代输出 | KenLM（PPL 计算） | eval_classifier（独立打分） | NB01-NB07 分析结果 |
| **训练阶段** | Wikipedia + Cosmopedia + CC WET 负样本 | 无 | — | 4 个分类器模型文件 |

---

### Part A：数据源

### 2.1 Pipeline 输入：Common Crawl WET

#### Common Crawl 数据分层

Common Crawl 对互联网进行周期性全量爬取，每次 Crawl 产出约 3-5B 网页。数据按处理程度分为三层：

| 层级 | 格式 | 内容 | 单 segment 大小 | 垃圾率 | 论文参考 |
|------|------|------|----------------|--------|---------|
| Level 1 | **WARC** | 完整 HTTP 响应（HTML + 头部） | ~1GB | ~70-85% | RefinedWeb 报告原始 HTML 噪声极高 |
| Level 2 | **WET** | 已提取的纯文本（无 HTML） | ~150-200MB | ~50-70% | FineWeb 论文：CC 文本平均质量低于已清洗语料 |
| Level 3 | FineWeb / Dolma | 经过 Gen1 级清洗的纯文本 | ~1GB/parquet | ~5-10% | FineWeb 论文：15T tokens 级别 |

#### 为什么选 CC WET（Level 2）？

| 备选方案 | 问题 |
|----------|------|
| WARC (Level 1) | 需要 HTML 解析（Trafilatura），本项目重点是质量过滤方法论对比，不是文本提取 |
| **CC WET (Level 2)** | 纯文本、垃圾率 ~50-70%，正好适合展示三代 pipeline 的过滤效果差异 |
| FineWeb (Level 3) | **已经过 Gen1 级清洗**，垃圾率仅 ~5-10%，在已清洗数据上跑清洗 pipeline 无法展示效果 |

#### 数据流：级联式架构

本项目采用级联式设计，Gen2/Gen3 的输入是 Gen1 的输出（而非原始 CC WET）：

```
CC WET 原始数据 (12K~100K 条)
    │
    ▼
  Gen1 Pipeline（规则过滤）
    │ 输出: gen1_output.jsonl
    ├─────────────────────┐
    ▼                     ▼
  Gen2 Pipeline         Gen3 Pipeline
  （分类器过滤）         （集成+路由+改写）
    │                     │
    ▼                     ▼
  gen2_output.jsonl     gen3_output.jsonl
```

> **为什么不让三代都从原始 CC WET 开始？** 这也是论文中的标准做法。DCLM 的分类器过滤建立在 Dolma（已做 Gen1 级清洗）的基础上；Nemotron-CC 的 Hybrid Pipeline 也包含了 heuristic 步骤。Gen1 的规则过滤是所有后续方法的"基础层"——先去除明显垃圾（非英文、乱码、重复），再在较干净的数据上应用更精细的方法。

#### 采样策略：多 Segment 随机采样

CC 每个 Crawl 包含约 90,000 个 segment。**不能从单个 segment 顺序读取**，否则会引入偏差：

| 偏差类型 | 原因 |
|----------|------|
| 域名集中 | 同一 segment 内相邻记录可能来自同一网站 |
| 语言偏斜 | 某些 segment 可能偏向特定语言区域 |
| 质量同质化 | 同一批爬取的页面质量相近 |

**解决方案**：从多个随机 segment 均匀采样，再全局 shuffle。

```
偏斜方案：1 个 segment × 12,000 条 = 12,000 （集中、可能偏斜）
均匀方案：10 个 segment × 1,200 条 = 12,000 （分散、更均匀）
```

---

### 2.2 分类器正样本 A：Wikipedia 摘要

#### 为什么用 Wikipedia 做正样本？

DCLM 论文（NeurIPS 2024）的核心思路：训练二分类器区分"高质量"和"低质量"文本，需要一个公认的高质量文本集合作为正样本。论文测试了两种正样本来源：

| 正样本来源 | DCLM 论文 7B MMLU | 特点 | 论文参考 |
|-----------|-------------------|------|---------|
| **Wikipedia** | 64.0% | 百科风格，事实密集，结构规范 | DCLM Table 3 |
| OpenHermes | 63.2% | GPT-4 生成的指令跟随数据 | DCLM Table 3 |
| Wikipedia + OpenHermes | 63.8% | 混合正样本 | DCLM Table 3 |

> **OpenHermes 简介**：OpenHermes 2.5 是由 Teknium 团队整理的约 100 万条指令-回答对数据集，内容由 GPT-4 生成，覆盖代码、数学、角色扮演、写作等多种任务类型。DCLM 用它作为"高质量"正样本的另一选择——代表的是"对 AI 有用的文本"而非"百科式规范文本"。其 MMLU 略低于 Wikipedia（63.2% vs 64.0%），原因可能是指令跟随数据的文体分布与 MMLU 测试的学科知识风格不完全匹配。

Wikipedia 单独使用效果最好，且无版权争议、可复现性强，是学术界的标准选择。

#### 本项目的样本选择

| 属性 | 值 | 说明 |
|------|-----|------|
| 数据来源 | HuggingFace `wikimedia/wikipedia` 20231101.en | 英文维基百科 2023-11 月版本快照 |
| 提取方式 | Streaming 读取，取每篇文章的**首段**（`text.split("\n\n")[0]`） | 首段通常是全文摘要，信息密度最高 |
| 过滤条件 | 首段长度 ≥ 100 字符 | 排除过短的消歧义页/重定向页 |
| 最终数量 | 5,000 条 | fastText 几千条即可收敛（DCLM 论文未指定最小量） |
| 文本特征 | 中位数 57 词（352 字符），最长 399 词 | 简洁的百科摘要，非完整文章 |

#### 已知偏斜与局限

**1. 话题分布偏斜**：Wikipedia 数据集按字母序存储，streaming 前 5,000 条以 A/B 开头为主（A: 26%, B: 16%），后半字母表条目不足。

**2. 文体单一**：全部是百科风格——客观陈述、第三人称、事实密集。导致分类器学到的"高质量"定义偏向此文体，对以下类型产生低估：
- 新闻报道（第一人称视角、时效性语言）
- 技术博客（代码片段、口语化表达）
- 学术论文（专业术语密集、引用格式）
- 对话/论坛（多人交互、非正式语言）

**3. 这恰好是 Gen2 分布偏移的根源**：正样本定义了"高质量"的边界。用百科做正样本 → 分类器偏好百科风格 → 保留数据多样性降低。这一局限正是 Gen3 需要集成多个分类器的动机（§1.3 创新 1）。

---

### 2.3 分类器正样本 B：Cosmopedia 教育类文本

#### 为什么需要第二种正样本？

Gen3 的分类器集成（§1.3）需要多个分类器具有**不同的正样本偏好**。Wikipedia 正样本训练的分类器偏好百科风格；教育类正样本训练的分类器偏好教科书风格。两者对"高质量"的定义边界不同，取并集可以扩大覆盖面。

#### 为什么选 Cosmopedia 而非 Nemotron-CC 的原始方案？

Nemotron-CC 论文的教育类分类器使用 467K FineWeb 文本 + LLM（Mixtral/Nemotron-4-340B）标注教育价值分来训练。本项目不采用这一方案，原因是**循环偏差风险**：

| 方案 | 训练数据 | 循环偏差风险 |
|------|---------|-------------|
| Nemotron-CC 原始方案 | FineWeb 文本 + LLM 标注 | **高**：FineWeb 本身是 CC 经 Gen1 清洗后的产物，用它训练分类器再去过滤 CC，等于间接引入 FineWeb 的过滤偏好 |
| **本项目方案** | Cosmopedia 合成教科书 | **无**：Cosmopedia 是 Mixtral-8x7B 基于主题提示词从零生成的合成文本，与 CC 数据完全无关 |

#### Cosmopedia 数据集简介

| 属性 | 值 |
|------|-----|
| 来源 | HuggingFace `HuggingFaceTB/cosmopedia`，openstax 子集 |
| 生成方式 | Mixtral-8x7B-Instruct 根据 OpenStax 教科书主题生成合成教育文本 |
| 总规模 | 全量 30M 样本 / 25B tokens（本项目仅取 5,000 条） |
| 内容风格 | 教科书段落——结构清晰、有教学目标、循序渐进的解释 |
| 文本特征 | 平均 ~3,800 字符/条，比 Wikipedia 摘要（~350 字符）长约 10 倍 |

#### 为什么选 openstax 子集？

Cosmopedia 包含多个子集（openstax、khanacademy、stanford、stories、wikihow 等）。选择 openstax 的原因：
- **最像真实教科书**：基于 OpenStax 开源教材主题生成，内容涵盖物理、化学、生物、经济等学科
- **与 Wikipedia 差异最大**：Wikipedia 是百科式事实陈述，OpenStax 是教学式渐进解释——两种正样本训练出的分类器偏好互补
- **单一子集足够多样**：OpenStax 覆盖多个学科，5,000 条已包含足够的主题多样性

#### 5,000 条是否足够？

足够。fastText 是轻量级模型，对训练样本量需求不高：
- Gen2 的 fasttext_dclm 使用同样 5,000 条 Wikipedia，在本项目中运行效果良好
- fastText 论文（Joulin et al. 2016）显示几百条每类即可收敛
- 关键瓶颈是正样本**多样性**（覆盖多少主题/文体）而非数量，Cosmopedia openstax 覆盖多学科，多样性充分

---

### Part B：模型

### 2.4 外部预训练模型

Pipeline 和评估依赖两个外部预训练模型，均不需要本项目训练，直接下载使用：

| 模型 | 文件路径 | 用途 | 使用阶段 | 来源 |
|------|---------|------|---------|------|
| **fastText lid.176.bin** | `data/models/lid.176.bin` | 语言检测 | **Gen1 Pipeline Step 2**（语言过滤） | Facebook Research |
| **KenLM Wikipedia EN** | `data/models/wikipedia/en.arpa.bin` + `en.sp.model` | Perplexity 计算 | **评估 Notebook**（NB02/NB05），不参与 Pipeline 过滤 | HuggingFace `edugp/kenlm` |

**fastText lid.176.bin**：支持 176 种语言识别（F1 > 0.95，Joulin et al. 2016）。输入一段文本，输出语言标签和置信度。本项目要求置信度 ≥ 0.65 才判为英文，低于此阈值的文档被过滤。C4 和 RefinedWeb 均使用类似的 fastText 语言检测方案。

**KenLM Wikipedia EN**：在英文 Wikipedia 上训练的 5-gram 语言模型，用于计算文本 Perplexity（困惑度）。Perplexity 越低表示文本越接近规范英文。**注意：KenLM 不参与任何 Pipeline 的过滤决策**，仅作为评估阶段的质量代理指标使用（NB02 中用于对比 Gen1 过滤前后的 PPL 分布）。该模型在 Wikipedia 上训练，因此也存在百科偏好——但作为评估工具，这种偏好是可接受的。

---

### 2.5 LLM API：Claude Opus

Gen3 Pipeline 的合成数据改写（创新 3，§1.3）调用外部 LLM API 对低质量文档进行改写。

| 属性 | 值 |
|------|-----|
| Provider | Anthropic |
| 模型 | `claude-opus-4-6`（最新 Opus，改写质量最高） |
| 配置文件 | `configs/api_config.yaml` |
| 使用阶段 | **Gen3 Pipeline Step 3**（低质量文档改写） |
| API Key | 存于 `.env` 文件（`FINEWEB_API_KEY`），不入库 |
| 调用规模 | 取决于 LQ 桶实际文档数（全量改写，无预算上限） |

> **为什么选 Opus 而非 Haiku？** 改写任务的目标是将低质量文本转化为高质量文本。更强的模型能更好地理解原文核心信息、消除噪声、提升表达质量。本项目的改写量级较小（百条级别），Opus 的成本完全可接受。
>
> **与 Nemotron-CC 的差异**：Nemotron-CC 使用 Mistral-7B-Instruct（开源模型本地部署），适合 TB 级数据的大规模改写。本项目规模较小，直接调用 API 更简单。

---

### 2.6 项目自行训练的模型

以下模型由项目代码自动训练（运行 `scripts/run_gen2.py` 和 `scripts/run_gen3.py` 时完成），不需要手动下载或预训练。训练原理和设计决策详见 §1.2 和 §1.3。

| 模型 | 训练脚本 | 正样本 | 超参数 | 使用阶段 | 保存路径 |
|------|---------|--------|--------|---------|---------|
| fasttext_dclm | `run_gen2.py` | Wikipedia | dim=64, wN=2 | Gen2 过滤 + Gen3 集成 | `results/quality_scores/gen2_classifier.bin` |
| fasttext_edu | `run_gen3.py` | Cosmopedia | dim=64, wN=2 | Gen3 集成 | `results/quality_scores/gen3_edu_classifier.bin` |
| tfidf_lr | `run_gen3.py` | Wikipedia | 50K features | Gen3 集成 | `results/quality_scores/gen3_tfidf_lr.pkl` |
| eval_classifier | `run_gen2.py` | Wikipedia | dim=32, wN=1 | **仅评估**（NB01-NB07） | `results/quality_scores/eval_classifier.bin` |

> **为什么 eval_classifier 要和 Pipeline 分类器用不同超参数？** 防止循环偏差——详见 §1.2「循环偏差防范」。

---

### 2.7 两档运行模式

通过 `configs/run_config.yaml` 的 `run_mode` 控制数据规模：

| 档位 | 读取文档数 | 数据文件 | 用途 |
|------|-----------|---------|------|
| `smoke_test` | 12,000 | `cc_wet_sample.jsonl` (10 segments) | 验证代码 + 产出有统计意义的对比数据 |
| `full_run` | 100,000 | `cc_wet_full.jsonl` (50 segments) | 最终展示级结果 |

In [1]:
# === 数据源概览：两档模式下的数据规模对比 ===
import sys, json, os, re
from pathlib import Path
from urllib.parse import urlparse
sys.path.insert(0, '..')

# ── 注册域名提取（取主域名，如 www.news.baidu.com → baidu.com）──
_MULTI_SUFFIXES = {
    "co.uk", "co.jp", "co.kr", "co.in", "co.nz", "co.za", "co.id",
    "com.au", "com.br", "com.cn", "com.hk", "com.tw", "com.mx", "com.sg",
    "org.uk", "org.au", "net.au", "ac.uk", "gov.uk", "edu.cn",
    "ne.jp", "or.jp", "ac.jp", "go.jp",
}

def get_registered_domain(netloc: str) -> str:
    host = netloc.split(":")[0].lower().strip(".")
    if not host:
        return ""
    parts = host.split(".")
    if len(parts) <= 2:
        return host
    two_suffix = ".".join(parts[-2:])
    if two_suffix in _MULTI_SUFFIXES:
        return ".".join(parts[-3:]) if len(parts) >= 3 else host
    return ".".join(parts[-2:])

# ── 加载两个数据文件 ──
sample_path = Path("../data/raw/cc_wet_sample.jsonl")
full_path = Path("../data/raw/cc_wet_full.jsonl")

def load_lines(path):
    if not path.exists():
        return []
    with open(path) as f:
        return f.readlines()

def compute_stats(lines):
    domains = set()
    segments = set()
    for line in lines:
        doc = json.loads(line)
        url = doc.get("url", "")
        try:
            domain = get_registered_domain(urlparse(url).netloc)
            if domain:
                domains.add(domain)
        except Exception:
            pass
        segments.add(doc.get("segment", "unknown"))
    return {"docs": len(lines), "domains": len(domains), "segments": len(segments)}

# ── 两档数据对比 ──
print("=" * 72)
print("两档运行模式 — 数据规模对比")
print("=" * 72)

sample_lines = load_lines(sample_path)
full_lines = load_lines(full_path)

tiers = [
    ("smoke_test",  12000,  sample_lines[:12000],   sample_path),
    ("full_run",    100000, full_lines[:100000],     full_path),
]

print(f"\n{'档位':<14} {'doc_limit':<11} {'实际文档':<11} {'独立域名':<11} {'域名/文档比':<12} {'segments':<10} {'数据文件'}")
print("-" * 100)
for mode, limit, lines, path in tiers:
    if not lines:
        print(f"{mode:<14} {limit:<11,} {'N/A':<11} {'N/A':<11} {'N/A':<12} {'N/A':<10} {path.name} (不存在)")
        continue
    stats = compute_stats(lines)
    ratio = f"{stats['domains']/stats['docs']:.1%}" if stats['docs'] else "N/A"
    print(f"{mode:<14} {limit:<11,} {stats['docs']:<11,} {stats['domains']:<11,} {ratio:<12} {stats['segments']:<10} {path.name}")

print(f"\n  注：域名按注册域名统计（www.news.example.com → example.com），去除子域名。")

# ── 文件大小对比 ──
print()
print("数据文件大小:")
for label, path in [("cc_wet_sample.jsonl", sample_path), ("cc_wet_full.jsonl", full_path)]:
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        with open(path) as f:
            line_count = sum(1 for _ in f)
        print(f"  {label}: {size_mb:.1f} MB, {line_count:,} 条")
    else:
        print(f"  {label}: 不存在")

# ── smoke_test 的 segment 分布 ──
if sample_lines:
    print()
    print("smoke_test segment 分布 (cc_wet_sample.jsonl):")
    seg_counts = {}
    for line in sample_lines:
        seg = json.loads(line).get("segment", "unknown")
        seg_counts[seg] = seg_counts.get(seg, 0) + 1
    for seg, cnt in sorted(seg_counts.items(), key=lambda x: -x[1]):
        print(f"  {seg}: {cnt:,}")

两档运行模式 — 数据规模对比



档位             doc_limit   实际文档        独立域名        域名/文档比       segments   数据文件
----------------------------------------------------------------------------------------------------
smoke_test     12,000      12,000      8,741       72.8%        10         cc_wet_sample.jsonl


full_run       100,000     100,000     63,933      63.9%        50         cc_wet_full.jsonl

  注：域名按注册域名统计（www.news.example.com → example.com），去除子域名。

数据文件大小:
  cc_wet_sample.jsonl: 81.3 MB, 12,000 条


  cc_wet_full.jsonl: 668.3 MB, 100,000 条

smoke_test segment 分布 (cc_wet_sample.jsonl):
  CC-MAIN-20240225171204-20240225201204-00598: 1,201
  CC-MAIN-20240223153350-20240223183350-00289: 1,201
  CC-MAIN-20240225071740-20240225101740-00456: 1,201
  CC-MAIN-20240221070402-20240221100402-00578: 1,201
  CC-MAIN-20240222193722-20240222223722-00834: 1,200
  CC-MAIN-20240305092259-20240305122259-00496: 1,200
  CC-MAIN-20240304165127-20240304195127-00110: 1,200
  CC-MAIN-20240302184020-20240302214020-00382: 1,200
  CC-MAIN-20240223021632-20240223051632-00192: 1,198
  CC-MAIN-20240226094435-20240226124435-00048: 1,198


In [2]:
# === 数据长什么样？展示 5 条典型样例 ===
import random, textwrap, statistics

# 使用上一 cell 的 sample_lines（cc_wet_sample.jsonl 全量）
all_lines = sample_lines
total_docs = len(all_lines)

PREVIEW_CHARS = 2000  # 每条文档展示的字符数（完整查看可改为更大值）

random.seed(42)
sample_indices = sorted(random.sample(range(total_docs), 5))

def sanitize(text):
    """清理 surrogate 字符，防止输出报错"""
    return re.sub(r'[\ud800-\udfff]', '', text)

print("=" * 70)
print("CC WET 原始数据样例（随机抽取 5 条，来自 cc_wet_sample.jsonl）")
print("=" * 70)

for i, idx in enumerate(sample_indices):
    doc = json.loads(all_lines[idx])
    text = sanitize(doc.get("text", ""))
    url = doc.get("url", "N/A")
    segment = doc.get("segment", "N/A")
    char_count = len(text)
    word_count = len(text.split())
    lines = text.strip().split("\n")

    print(f"\n{'─' * 70}")
    print(f"样例 {i+1} (index={idx})")
    print(f"  URL:     {url}")
    print(f"  Segment: {segment}")
    print(f"  字符数:  {char_count:,}  |  词数: {word_count:,}  |  行数: {len(lines)}")
    print(f"  内容（前 {PREVIEW_CHARS:,} 字符）:")
    preview = text[:PREVIEW_CHARS].replace('\n', '\n    ')
    print(f"    {preview}")
    if char_count > PREVIEW_CHARS:
        print(f"    ... (省略 {char_count - PREVIEW_CHARS:,} 字符)")
    print()

# 数据质量初印象
print("=" * 70)
print("数据质量初印象（cc_wet_sample.jsonl 全量统计）")
print("=" * 70)
char_lens = []
word_lens = []
lang_hints = {"en": 0, "non_en": 0}

for line in all_lines:
    doc = json.loads(line)
    text = doc.get("text", "")
    char_lens.append(len(text))
    word_lens.append(len(text.split()))
    # 粗略语言估计：检查前 500 字符中 ASCII 字母占所有字母的比例
    # 这不是精确的语言检测（精确检测在 Gen1 的 language_filter 中用 fastText langid 做）
    ascii_alpha = sum(1 for c in text[:500] if c.isascii() and c.isalpha())
    total_alpha = sum(1 for c in text[:500] if c.isalpha()) or 1
    if ascii_alpha / total_alpha > 0.5:
        lang_hints["en"] += 1
    else:
        lang_hints["non_en"] += 1

print(f"  文档字符数: 中位数={statistics.median(char_lens):,.0f}, 均值={statistics.mean(char_lens):,.0f}, P10={sorted(char_lens)[int(0.1*len(char_lens))]:,}, P90={sorted(char_lens)[int(0.9*len(char_lens))]:,}")
print(f"  文档词数:   中位数={statistics.median(word_lens):,.0f}, 均值={statistics.mean(word_lens):,.0f}")
print(f"  英文估计:   ~{lang_hints['en']/total_docs:.0%} ({lang_hints['en']:,} 条)  |  非英文: ~{lang_hints['non_en']/total_docs:.0%} ({lang_hints['non_en']:,} 条)")
print(f"    ↑ 粗略估计（ASCII 字母占比 > 50% 判为英文），精确语言检测在 Gen1 language_filter 步骤中用 fastText langid 完成。")
print(f"\n  说明: 非英文占比高是 CC WET 的正常特征（全球爬取数据），语言过滤步骤会处理。")

CC WET 原始数据样例（随机抽取 5 条，来自 cc_wet_sample.jsonl）

──────────────────────────────────────────────────────────────────────
样例 1 (index=409)
  URL:     http://angel3829.synology.me/xe/index.php?mid=board_dKNa22&page=10&document_srl=37627&listStyle=viewer
  Segment: CC-MAIN-20240302184020-20240302214020-00382
  字符数:  3,600  |  词数: 497  |  行数: 51
  内容（前 2,000 字符）:
    e Memo - Nutritional supplements are products intended for enhance
    Nutritional supplements are products intended for enhance
    by ylylaf posted Apr 30, 2023
    ?
    단축키
    Prev이전 문서
    Next다음 문서
    ESC닫기
    가 + - Up Down Comment Print
    Supplements constitute items designed to enhance an individual's regular consumption of nutrients. These items are available within a range of categories, for example capsules, granules, beverages, and chewables. The key target behind these types of dietary supplements is usually to be able to give the body necessary nutrients which may not become sufficiently attained by means of e

  文档字符数: 中位数=2,343, 均值=5,129, P10=360, P90=9,534
  文档词数:   中位数=313, 均值=717
  英文估计:   ~64% (7,660 条)  |  非英文: ~36% (4,340 条)
    ↑ 粗略估计（ASCII 字母占比 > 50% 判为英文），精确语言检测在 Gen1 language_filter 步骤中用 fastText langid 完成。

  说明: 非英文占比高是 CC WET 的正常特征（全球爬取数据），语言过滤步骤会处理。


## 附录：核心概念术语表

> 本节为查阅用参考资料。正文中以"详见术语表第 X 条"引用。

### 1. Quality-Quantity Trade-off
**定义**：过滤越激进，数据质量越高，但数据量越少。两者存在张力。

**DCLM 的发现**：top-10% 过滤使 MMLU 达到 64%，但丢弃了 90% 的数据。

**第三代的突破**：Nemotron-CC 证明这个 trade-off 可以被打破——通过分类器集成和数据改写，在保留 4倍数据的前提下质量不降。

---

### 2. Token Yield（Token 产出率）
**定义**：每处理 1GB 原始数据能产出多少高质量 tokens。

**重要性**：这是比"过滤率"更本质的指标。Nemotron-CC 用 Token Yield 来衡量不同 pipeline 的效率。不同方法的 Token Yield 差异可达 4-10 倍。

**例子**：处理同一批 CC 数据，第二代（top-10%）的 Token Yield = 第三代的 1/4。

---

### 3. Heuristic Filter 的不可组合性
**定义**：10 个 filter 各过滤 5%，总过滤率不是 50%。

**原因一（重叠）**：同一条数据可能被多个 filter 同时命中。

**原因二（顺序依赖）**：先过滤短文档再统计 N-gram，和先统计 N-gram 再过滤短文档，结果不同——后一个会导致 N-gram 统计基于更完整的语料。

**实践意义**：FineWeb 的过滤顺序经过精心设计，不能随意调整。

---

### 4. 代理指标 vs 端到端指标
**代理指标（Proxy Metrics）**：quality 分类器打分、Perplexity、N-gram 多样性——衡量数据本身特征。便宜但间接。

**端到端指标（End-to-End）**：Proxy Model 在 benchmark（MMLU/HellaSwag）的分数——直接衡量数据对模型的影响。可靠但昂贵（需要训练模型）。

**本项目策略**：主要使用代理指标（快速迭代），可选 Proxy Model 验证（Notebook 09）。

---

### 5. Chinchilla Scaling Law
**核心结论**：计算最优的训练配比是 tokens ≈ 参数量 × 20。

**含义**：125M 参数模型需要约 2.5B tokens 才能达到 Chinchilla 最优。7B 模型需要约 140B tokens。

**Token Horizon（训练 Token 总量）**：模型预训练实际消耗的总 token 数。例如 Llama 3 的 Token Horizon 为 15T tokens——远超 Chinchilla 最优点（7B × 20 = 140B），属于"过训练"（overtrain）策略：用更多数据换取更好的推理性能。现代大模型普遍采用这种策略，使得数据量成为核心瓶颈。

**对数据工程的启示**：第二代（top-10%）的数据量限制了它只适合短 Token Horizon 训练。长 Token Horizon（如 Llama 3 的 15T）必须要第三代这样的高 Token Yield 方法，否则数据不够用。

---

### 6. 循环偏差（Circular Bias）
**定义**：用同一个模型既过滤数据又评估数据质量，产生"自证"的假高分。

**具体案例**：用 fastText(OpenHermes 正样本)过滤后，再用同一个 fastText 打分，分必然很高——但这只说明分类器自洽，不说明数据真的好。

**解决方案**：评估分类器必须与 Pipeline 分类器独立训练（不同正样本、不同超参）。

---

### 7. Distribution Shift（分布偏移）
**定义**：过滤不仅改变数据量，还改变数据分布。

**典型案例**：激进质量过滤 → 百科/学术内容富集 → MMLU 分高，但对话/创作能力下降。

**量化方法**：用 N-gram unique ratio 和域名 Shannon Entropy 跟踪分布变化。

---

### 8. Data Mixing（数据配比）
**定义**：清洗后将不同来源的数据按比例混合（Web + Code + Books + Wiki）。

**Llama 3 的配比（参考）**：Web 50% + Code 25% + 学术 15% + Wikipedia 5% + 其他 5%。

**本项目定位**：我们聚焦"清洗过滤"这一步，Data Mixing 是下游环节。